In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
import os
from PIL import Image

import torch
from torchvision.transforms import v2
from torchvision.io import decode_image

from scripts.prepare import copy_directory_structure, get_valid_paths, process_dataframes, get_plot_histograms, resample
from scripts.preprocessing import get_class_weights, get_value_weights, run_pipeline, pipeline

# Testing Environment Setup

If you simply want to use the models to obtain inferences, this notebook skips the up-front image processing steps, and instead images will be processed directly from the source file whenever they are referenced. However, this may create significantly more computational overhead, slowing down model training immensely.

**Actions performed in this notebook:**

1) Preprocessing the label data frames to create the train/test/validation datasets
2) Perform resampling on the training dataset


## 1. Preprocess the Label Data Frames

These source_roots are the paths to the folders containing the respective  patient image folders downloaded  from the blob and organized as specified in the data dictionary.

In [2]:
# # --- Input directories
# source_train_root1 = "data/raw_data/CheXpert-v1.0 batch 2 (train 1)/"
# source_train_root2 = "data/raw_data/CheXpert-v1.0 batch 3 (train 2)/"
# source_train_root3 = "data/raw_data/CheXpert-v1.0 batch 4 (train 3)/"
# source_valid_rad_root = "data/raw_data/CheXpert-v1.0 batch 1 (validate & csv)/valid/"
# source_test_rad_root = "data/raw_data/test/"

# # --- Output directories
# train_valid_root = "data/processed_data/train_valid/"
# test_root = "data/processed_data/test/"
# valid_rad_root = "data/processed_data/valid_rad/" 
# test_rad_root = "data/processed_data/test_rad/"

# --- Input directories
source_train_root1 = "raw_data/CheXpert-v1.0 batch 2 (train 1)/"
source_train_root2 = "raw_data/CheXpert-v1.0 batch 3 (train 2)/"
source_train_root3 = "raw_data/CheXpert-v1.0 batch 4 (train 3)/"
source_valid_rad_root = "raw_data/CheXpert-v1.0 batch 1 (validate & csv)/valid/"
source_test_rad_root = "raw_data/test/"

# --- Output directories
train_valid_root = "data/train_valid/"
test_root = "data/test/"
valid_rad_root = "data/valid_rad/" 
test_rad_root = "data/test_rad/"

## 2. Preprocess the Label Data Frames

The training set will be from a sample of 80% of instances from batch 2 and 3<br>
The validation set will be from the other 20% of instances from batch 2 and 3<br>
The test set will be batch 4, the smallest of the training datasets

The valid rad (radiologist) and test rad sets are not primarily used, but will serve as how our model performs against real human annotations

**Actions taken to process the data frames:**
1) Loop through the specified source_root folder to find all image paths that exist
   1) Store each path string in the valid_paths list
2) For each image metadata row in df, add new path columns:
   1) ```floating_file_path```: The image file path without the root folder (ex: ```S:/Root/patientX/image.jpg``` to ```patientX/image.jpg```)
   2) ```source_file_path```: The file path to the original image saved from the blob
   3) ```base{dim}_file_path```: The file path to the saved dim x dim processed image
   4) ```base{dim}_file_path2```: The file path to the saved enhanced dim x dim processed image
3) Filter out any images whose source_file_path is not contained in the valid_paths (created in step 1)
4) Return the processed dataframes

### Load Data

In [3]:
# --- Image sizes
dims = [224, 384]

# --- Original roots from the input files
old_root = "CheXpert-v1.0/train/"
old_test_root = "test/"

# --- Input files
# labels = "data/raw_data/train_cheXbert.csv"
# valid_rad_labels = "data/raw_data/CheXpert-v1.0 batch 1 (validate & csv)/valid.csv"
# test_rad_labels = "data/raw_data/test_labels.csv"
labels = "raw_data/train_cheXbert.csv"
valid_rad_labels = "raw_data/CheXpert-v1.0 batch 1 (validate & csv)/valid.csv"
test_rad_labels = "raw_data/test_labels.csv"

# --- Load the training/validation csvs
data_df = pd.read_csv(labels)
valid_rad_df = pd.read_csv(valid_rad_labels)
test_rad_df = pd.read_csv(test_rad_labels)

print(f"# rows in data_df: {len(data_df)}")
print(f"# rows in valid_rad_df: {len(valid_rad_df)}")
print(f"# rows in test_rad_df: {len(test_rad_df)}")

# rows in data_df: 223414
# rows in valid_rad_df: 234
# rows in test_rad_df: 668


### Train/Test/Validation Splits

In [4]:
# --- Split the df into the corresponding train/valid/test sets
train_df1 = process_dataframes(data_df, source_train_root1, old_root, train_valid_root, dims)
train_df2 = process_dataframes(data_df, source_train_root2, old_root, train_valid_root, dims)
# Combine the train_dfs
train_df0 = pd.concat([train_df1, train_df2], axis=0)
# Split into corresponding train/validation sets
train_df, valid_df = train_test_split(train_df0, test_size=0.2)


In [5]:
# --- Process the test, valid rad, and test rad sets
test_df = process_dataframes(data_df, source_train_root3, old_root, test_root, dims)
#
valid_rad_df = process_dataframes(valid_rad_df, source_valid_rad_root, old_root, valid_rad_root, dims)
test_rad_df = process_dataframes(test_rad_df, source_test_rad_root, old_test_root, test_rad_root, dims)

In [6]:
print(f"# rows in train_df: {len(train_df)}")
print(f"# rows in valid_df: {len(valid_df)}")
print(f"# rows in test_df: {len(test_df)}")
print(f"# rows in valid_rad_df: {len(valid_rad_df)}")
print(f"# rows in test_rad_df: {len(test_rad_df)}")

# rows in train_df: 145949
# rows in valid_df: 36488
# rows in test_df: 40977
# rows in valid_rad_df: 234
# rows in test_rad_df: 668


### Filter out unnecessary columns and rows

In [7]:
metadata_cols = ["Path", "Sex", "Age", "Frontal/Lateral", "AP/PA"]
filepath_cols = ["floating_file_path", "source_file_path", 
                 "base224_file_path", "base224_file_path2", 
                 "base384_file_path", "base384_file_path2"]

# Keep only these conditions
class_cols = ["Cardiomegaly", "Consolidation", "Edema", "Atelectasis", "Pleural Effusion"]
n_classes = len(class_cols)

# Filter out for only the class cols of interest
train_df = train_df.loc[:, class_cols + metadata_cols + filepath_cols]
valid_df = valid_df.loc[:, class_cols + metadata_cols + filepath_cols]
test_df = test_df.loc[:, class_cols + metadata_cols + filepath_cols]
valid_rad_df = valid_rad_df.loc[:, class_cols + metadata_cols + filepath_cols]
test_rad_df = test_rad_df.loc[:, class_cols + ["Path"] + filepath_cols]

In [8]:
### Filter for only Frontal views
train_df = train_df[train_df["Frontal/Lateral"]=="Frontal"].reset_index(drop=True)
valid_df = valid_df[valid_df["Frontal/Lateral"]=="Frontal"].reset_index(drop=True)
test_df = test_df[test_df["Frontal/Lateral"]=="Frontal"].reset_index(drop=True)
valid_rad_df = valid_rad_df[valid_rad_df["Path"].str.find("frontal") > 0].reset_index(drop=True)
test_rad_df = test_rad_df[test_rad_df["Path"].str.find("frontal") > 0].reset_index(drop=True)


In [9]:
### Map the values to 0/1/2 for CrossEntropyLoss usage
for col in class_cols:
    train_df[col] = train_df[col].map(lambda x: 1 if x==1 else 2 if x==-1 else 0)
    valid_df[col] = valid_df[col].map(lambda x: 1 if x==1 else 2 if x==-1 else 0)
    test_df[col] = test_df[col].map(lambda x: 1 if x==1 else 2 if x==-1 else 0)

In [10]:
### Remove rows from the training set where all class columns are zero
train_df_filtered = train_df[~train_df.index.isin(train_df[(train_df[class_cols]==0).all(axis=1)].index)]

## 3. Perform Weighted Resampling to Address Class Imbalances

### Sample 30,000 rows with Weighted Replacement

**Pseudo-code**:

```
for i in range(30,000):
    Calculate the sum of each class value (0, 1, or 2) for each column and store as a matrix
    For each class:
        Calculate the current class value proportions
        Invert the proportions to get higher weights for less-represented class values for that condition
        Process the weights so that they add up to one for the condition
```

In [11]:
%%time
input_df = train_df_filtered.copy()
n_samples = 30000
#
train_df_30000 = resample(input_df, class_cols, n_samples)
#
get_plot_histograms(train_df_30000, class_cols, f"{n_samples} Weighted Samples")

NameError: name 'output_df' is not defined

#### Downsample 30,000 rows randomly from the filtered training dataset

This is to reduce the overall counts of negative instances to make the resampling more effective

In [12]:
train_df_filtered2 = train_df_filtered.copy().sample(len(train_df_filtered) - 30000)

#### Combine the two extracted data frames

In [13]:
train_df_oversampled = pd.concat([train_df_filtered2, train_df_30000], axis=0).reset_index(drop=True)

NameError: name 'train_df_30000' is not defined

#### View the differences in class proportions

#### View details on the finalized datasets

In [ ]:
print(f"# rows in train_df: {len(train_df)}")
print(f"# rows in train_df_30000: {len(train_df_30000)}")
print(f"# rows in train_df_oversampled: {len(train_df_oversampled)}")
print(f"# rows in valid_df: {len(valid_df)}")
print(f"# rows in test_df: {len(test_df)}")
print(f"# rows in valid_rad_df: {len(valid_rad_df)}")
print(f"# rows in test_rad_df: {len(test_rad_df)}")

In [ ]:
train_df.head(5)

#### Save the files as csv

In [ ]:
### Save new training, validation, test csvs
train_df.to_csv("data/processed data/train_df.csv", index=False)
train_df_30000.to_csv("data/processed data/train_df_30000.csv", index=False)
train_df_oversampled.to_csv("data/processed data/train_df_oversampled.csv", index=False)
#
valid_df.to_csv("data/processed data/valid_df.csv", index=False)
test_df.to_csv("data/processed data/test_df.csv", index=False)
#
valid_rad_df.to_csv("data/processed data/valid_rad_df.csv", index=False)
test_rad_df.to_csv("data/processed data/test_rad_df.csv", index=False)

# Image Preprocessing Pipelines

The following cells show the image pipeline in action:


In [ ]:
# --- Filepaths for dfs
### train_df_oversampled is used to minimize excess image conversions for unused training images
train_df_filepath = "data/train_df_oversampled.csv"
#
valid_df_filepath = "data/valid_df.csv"
test_df_filepath = "data/test_df.csv"
#
valid_rad_df_filepath = "data/valid_rad_df.csv"
test_rad_df_filepath = "data/test_rad_df.csv"

# --- Image size --> Include the desired image sizes in this list
dim = 224
dims = [224]

# --- Set seed for reproducibility (optional)
seed = 100
if seed:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
# Default image enhancement variables
pipeline_kwargs = dict(
    scale_min=0,
    scale_max=255,
    usm_sigma=10,
    usm_weight=1.2,
    he_sigma=5
)

In [ ]:
# Load the training/validation csvs
train_df = pd.read_csv(train_df_filepath)
valid_df = pd.read_csv(valid_df_filepath)
test_df = pd.read_csv(test_df_filepath)
valid_rad_df = pd.read_csv(valid_rad_df_filepath)
test_rad_df = pd.read_csv(test_rad_df_filepath)

### Preprocessing steps for raw images:

* Scale the image values to the range [0-255]
* Resize the training, validation, and test images to get (224x224) and (384x384)
* Convert the arrays to type uint8 for compatibility with Image
* Save the processed images as jpeg files

In [ ]:
print("These examples show the images going through the initial pipeline")
print("These will be saved ahead of time to save time later with modelling")
print()

### Examples of images going through the pipeline
output_df = train_df.copy()
n_instances = len(output_df)

input_paths = output_df.loc[np.random.choice(range(1000), 3), "source_file_path"]
#
for i,input_file_path in enumerate(input_paths, start=1):
    print(f"Image {i} ---> rotation90_angle = 0, rotation_angle = 0")
    #
    fig,ax = plt.subplots(1,3, figsize=(7,3))
    imgs, img_enhanceds = pipeline(input_file_path, dims, **pipeline_kwargs)
    #
    with Image.open(input_file_path) as raw_img:
        ax[0].imshow(raw_img, cmap="gray")
        ax[0].set(title="Source Image")
        ax[0].axis("off")
    #
    ax[1].imshow(imgs[0], cmap="gray")
    ax[1].set(title="Raw Image")
    ax[1].axis("off")
    #
    ax[2].imshow(img_enhanceds[0], cmap="gray")
    ax[2].set(title="Enhanced Image")
    ax[2].axis("off")
    plt.show()


### Preprocessing steps for enhanced images

* Scale the image values to the range [0-255]
* Resize the training, validation, and test images to get (224x224) and (384x384)
* Sharpen the images using unsharp masking
* Equalize the histograms to increase contrast
* Convert the arrays to type uint8 for compatibility with Image
* Save the processed images as jpeg files

In [ ]:
print("These examples show the preprocessing performed in the dataloader during fitting")
print("These added transformations add randomness to account for oversampling")
print()
print("Without these transformations, the model is more likely to be biased toward duplicate samples")
print()
print("A random center crop is added to help the model differentiate the lung from the background")
print()
### Visualizing the image transformations from the dataloader 
# This function is within the dataloader, but is copied here for direct reference
def random_preprocess(img, dim, rot90_k, random_float):
    img = torch.tensor(img).unsqueeze(0)
    # Randomly rotate in a 90 degree interval
    img = torch.rot90(img, rot90_k, (1,2))
    #
    img = v2.functional.rotate(img, random_float[0]*18, interpolation=v2.InterpolationMode.BILINEAR)
    # Center crop and resize
    if np.abs(random_float[1]) > 1.5:
        img = v2.CenterCrop(dim - 48)(img)
        img = v2.Resize((dim,dim), interpolation=v2.InterpolationMode.BILINEAR)(img)
    if np.abs(random_float[1]) > 1:
        img = v2.CenterCrop(dim - 32)(img)
        img = v2.Resize((dim,dim), interpolation=v2.InterpolationMode.BILINEAR)(img)
    elif np.abs(random_float[1]) > 0:
        img = v2.CenterCrop(dim - 16)(img)
        img = v2.Resize((dim,dim), interpolation=v2.InterpolationMode.BILINEAR)(img)
    return img

In [ ]:
#####
output_df = train_df.copy()
n_examples = 5
#
n_instances = len(output_df)
input_paths = output_df.loc[np.random.choice(range(1000), 5), "source_file_path"]

for i,input_file_path in enumerate(input_paths, start=1):
    # Set rotation parameters
    rot90_k = np.random.choice(range(n_examples), n_instances, replace=True)
    random_float = np.random.randn(n_examples,2)
    # Process images
    imgs, _ = pipeline(input_file_path, dims, **pipeline_kwargs)
    imgs = [random_preprocess(imgs[0], dim, rot90_k=rot90_k[j], random_float=random_float[j,:]) for j in range(n_examples)]
    #
    fig,axes = plt.subplots(1,n_examples+1, figsize=(9,2.5))
    with Image.open(input_file_path) as raw_img:
        axes[0].imshow(raw_img, cmap="gray")
        axes[0].set(title=f"Raw Image")
        axes[0].axis("off")
    axes[3].set(title="Example Random Preprocessing")
    for img,ax in zip(imgs, axes.flatten()[1:]):
        img = img.numpy().astype(np.uint8)[0,:,:]
        with Image.open(input_file_path) as raw_img:
            ax.imshow(img, cmap="gray")
            ax.axis("off")
    #
    # ax[1].imshow(img, cmap="gray")
    # ax[1].set(title="Input to ResNet")
    # ax[1].axis("off")

    plt.show()

